# Quantum cosmology: the loop bounce

*cosmology · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/cosmology/quantum_gravity_toy)

A toy model of quantum gravity: loop quantum cosmology (LQC) of a flat universe filled with a massless scalar field. Replacing the connection $b$ by the holonomy $\sin(\lambda b)/\lambda$ of a loop of Planck-size area turns the Friedmann equation into $H^2 = \frac{8\pi G}{3}\rho(1-\rho/\rho_c)$. RK4 on a batch of universes shows the Big Bang replaced by a bounce at $\rho = \rho_c \approx 0.41\,\rho_\mathrm{P}$, while the Wheeler–DeWitt wave packet of the same model follows classical collapse into the singularity. Quantum gravity is an open research problem; this is one simplified approach.

**Vector form**

$$C = \frac{p_\varphi^2}{2V} - \frac{3V\sin^2(\lambda b)}{8\pi G\gamma^2\lambda^2} = 0,\qquad \dot V = -4\pi G\gamma\,\frac{\partial C}{\partial b},\quad \dot b = 4\pi G\gamma\,\frac{\partial C}{\partial V},\quad \dot\varphi = \frac{p_\varphi}{V},\qquad H^2 = \frac{8\pi G}{3}\,\rho\Big(1 - \frac{\rho}{\rho_c}\Big),\qquad \partial_\varphi^2\Psi = 12\pi G\,\partial_{\ln V}^2\Psi$$

**Index (tensor) form**

$$\dot y_{bk} = f_k(y_b;\,p_b,\lambda_b),\qquad y_{bk} = (V_b, b_b, \varphi_b),\qquad \rho_{tb} = \frac{p_b^2}{2V_{tb}^2},\qquad H_{tb} = \frac{\sin(2\lambda_b b_{tb})}{2\gamma\lambda_b},\qquad \Psi_{px} = A_k\,e^{\,i k (x_x - x_B + \ln 2 + \sqrt{12\pi G}\,\varphi_p)}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $y_{bk}$ | `initial_state(p, lam, rho0)` | (B, 3) | volume, connection and scalar field of each universe $b$ in the batch, on the constraint surface $C = 0$ |
| $\lambda_b$ | `LAMBDA * scales` | (B,) | loop size: $\lambda^2 = 4\sqrt3\pi\gamma\,\ell_P^2$ is the LQG area gap; smaller $\lambda$ ⇒ larger $\rho_c$ |
| $f_k(y_b)$ | `rhs(y, p, lam)` | (B, 3) | Hamilton's equations of the effective constraint, all universes at once |
| $y_{tbk}$ | `integrate(rhs, y0, dt, n)` | (T, B, 3) | RK4 trajectory in proper time |
| $C_{tb}$ | `constraint(y, p, lam)` | (T, B) | Hamiltonian constraint, must stay $0$ (its RK4 drift is printed in the terminal) |
| $\rho_c$ | `critical_density(lam)` | (B,) | $3/(8\pi G\gamma^2\lambda^2)$: the bounce density |
| $\Psi_{px}$ | `einsum('k,pxk->px', A, waves)` | (P, X) | repeated $k$ ⇒ summed: Wheeler–DeWitt packet on the grid of clock $\varphi_p$ and $x = \ln V$ |

Planck units $G = \hbar = c = 1$; Barbero–Immirzi parameter $\gamma = 0.2375$. Which approach: the effective equations of loop quantum cosmology with improved ($\bar\mu$) dynamics (Ashtekar, Pawlowski and Singh 2006; Taveras 2008), for a spatially flat FRW universe with a massless scalar field that also serves as a clock. Status: there is no accepted theory of quantum gravity. LQC is a symmetry-reduced quantisation inspired by loop quantum gravity; its link to full LQG is not established, and the effective equations are an approximation that agrees with the numerical evolution of sharply peaked quantum states in this model. No observation has tested the bounce. What is shown is exact for the model: the constraint is preserved, the effective Friedmann equation holds, the bounce density $\rho_c$ is the same for every $p_\varphi$, and the volume follows $V(t) = \sqrt{V_B^2 + 12\pi G p_\varphi^2 t^2}$. Far from $\rho_c$ (or for $\lambda \to 0$) classical general relativity returns: $H^2 = 8\pi G\rho/3$ and the volume falls linearly to $V = 0$ in finite time (red dashed: the singularity). The grey band is an exact Wheeler–DeWitt solution of the same model (quantised without loops, $x = \ln V$): it is a free wave packet that slides along the classical contracting line into $V \to 0$, as APS found. The lighter curves use smaller loops ($\lambda/2$, $\lambda/4$): the bounce happens later, at higher density. Frames are spaced densely around the bounce.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/cosmology/quantum_gravity_toy/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    scales = params["gap_scale"] * torch.tensor([1.0, 0.5, 0.25], dtype=torch.float64, device=dev)
    lam = tensors.LAMBDA * scales  # (B,) three loop sizes: ρ_c × 1, 4, 16
    p = torch.full_like(lam, params["p_phi"])
    rho_c = equation.critical_density(lam)
    y0 = tensors.initial_state(p, lam, 10 ** params["log10_rho_start"] * float(rho_c[0]))
    t_bounce = equation.time_to_bounce(y0, p, lam)
    sol = solve.evolve(y0, p, lam, params["dt"], 2 * float(t_bounce.max()))  # y: (T, B, 3)

    y = sol["y"]
    rho, H = equation.density(y, p), equation.hubble(y, lam)
    C = equation.constraint(y, p, lam) / equation.matter_energy(y, p)
    print(
        f"Planck units. γ = {tensors.GAMMA}, area gap λ² = {tensors.LAMBDA**2:.3f} ℓ_P², p_φ = {params['p_phi']:.0f} ħ"
    )
    for j in range(len(lam)):
        V_min = float(y[:, j, 0].min())
        V_B = float(p[j] / torch.sqrt(2 * rho_c[j]))
        ratio = float(rho[:, j].max() / rho_c[j])
        print(f"λ × {float(scales[j]):5.3f}: ρ_c = {float(rho_c[j]):8.3f} ρ_P   max sampled ρ/ρ_c = {ratio:.6f}"
              f"   V_min = {V_min:9.2f} (V_B = {V_B:9.2f})")  # fmt: skip
    print(f"max |C| / (p_φ²/2V) along all trajectories: {float(C.abs().max()):.1e} (constraint preserved)")
    t_sing = equation.classical_singularity_time(y0[0, 0], p[0])
    print(f"classical GR from the same start hits V = 0 after t = {t_sing:.3f} t_P; "
          f"LQC bounces at t = {float(t_bounce[0]):.3f}")  # fmt: skip

    # Wheeler–DeWitt packet in the (φ, ln V) plane, aligned with the bounce of the first universe
    phi = y[:, 0, 2]
    phi_b = float(phi[y[:, 0, 0].argmin()])
    x_b = math.log(float(p[0] / torch.sqrt(2 * rho_c[0])))
    x, phi_grid = tensors.wdw_grid(x_b - 2, float(torch.log(y[:, 0, 0]).max()) + 0.5, float(phi[0]), float(phi[-1]),
                                   260, 180, dev)  # fmt: skip
    psi = equation.wdw_packet(x, phi_grid - phi_b, float(p[0]), x_b, spread=params["spread"])  # (P, X)
    print("Wheeler–DeWitt packet: follows the classical line ln V = ln V_B − ln 2 − √(12πG)(φ − φ_B) into V → 0")
    return plot.animate(sol, p, lam, scales, rho_c, H, rho, (x, phi_grid, psi, x_b, phi_b),
                        out_dir / "quantum_gravity_toy.gif")  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/cosmology/quantum_gravity_toy/equation.py
r"""Physics → tensor translation: effective loop quantum cosmology of a flat FRW universe with a massless scalar.

Classical GR (flat FRW, massless scalar φ with momentum p_φ, cell volume V = a³): the Hamiltonian constraint is

    C_GR = p_φ²/(2V) − 3 V b²/(8πGγ²) = 0,       H = b/γ   ⇒   H² = (8πG/3) ρ,   ρ = p_φ²/(2V²).

Loop quantum cosmology replaces the connection b by holonomies around a loop of the smallest LQG area λ² = Δ,
b → sin(λb)/λ (Ashtekar–Pawlowski–Singh 2006). The *effective* constraint (an approximation that tracks sharply
peaked quantum states, checked numerically by APS) is

    C = p_φ²/(2V) − 3 V sin²(λb) / (8πGγ²λ²) = 0.

Hamilton's equations with {b, V} = 4πGγ and lapse 1 (proper time t), for every batch member b at once:

    V̇ = −4πGγ ∂C/∂b = 3 V sin(λb) cos(λb) / (γλ)       ⇒ H = V̇/(3V) = sin(2λb)/(2γλ)
    ḃ =  4πGγ ∂C/∂V = 4πGγ [ −p_φ²/(2V²) − 3 sin²(λb)/(8πGγ²λ²) ]
    φ̇ = p_φ / V

On C = 0 this gives the effective Friedmann equation H² = (8πG/3) ρ (1 − ρ/ρ_c), ρ_c = 3/(8πGγ²λ²): the Big Bang
is replaced by a bounce at ρ = ρ_c. The exact solution is V(t) = √(V_B² + 12πG p_φ² t²), V_B = p_φ/√(2ρ_c).

Wheeler–DeWitt (the same model quantised without loops): with x = ln V the constraint becomes a free Klein–Gordon
equation in the clock φ, ∂²_φ Ψ = 12πG ∂²_x Ψ, solved exactly by a superposition of plane waves.
"""

import math

import torch

from .tensors import GAMMA, G


def critical_density(lam: torch.Tensor) -> torch.Tensor:
    """ρ_c = 3/(8πGγ²λ²); λ → 0 sends it to infinity (classical limit)."""
    return 3 / (8 * math.pi * G * GAMMA**2 * lam**2)


def density(y: torch.Tensor, p_phi: torch.Tensor) -> torch.Tensor:
    """ρ = p_φ²/(2V²), the scalar's energy density.   y (..., B, 3) → (..., B)"""
    return p_phi**2 / (2 * y[..., 0] ** 2)


def matter_energy(y: torch.Tensor, p_phi: torch.Tensor) -> torch.Tensor:
    """p_φ²/(2V): the matter term of the constraint (energy in the cell)."""
    return p_phi**2 / (2 * y[..., 0])


def constraint(y: torch.Tensor, p_phi: torch.Tensor, lam: torch.Tensor) -> torch.Tensor:
    r"""C = p_φ²/(2V) − 3V sin²(λb)/(8πGγ²λ²), which must stay 0.   → (..., B)"""
    V, b = y[..., 0], y[..., 1]
    return p_phi**2 / (2 * V) - 3 * V * torch.sin(lam * b) ** 2 / (8 * math.pi * G * GAMMA**2 * lam**2)


def hubble(y: torch.Tensor, lam: torch.Tensor) -> torch.Tensor:
    """H = sin(2λb)/(2γλ) → b/γ as λ → 0."""
    return torch.sin(2 * lam * y[..., 1]) / (2 * GAMMA * lam)


def rhs(y: torch.Tensor, p_phi: torch.Tensor, lam: torch.Tensor) -> torch.Tensor:
    """dy/dt from Hamilton's equations, all batch members at once.   y (B, 3) → (B, 3)"""
    V, b = y[:, 0], y[:, 1]
    s, c = torch.sin(lam * b), torch.cos(lam * b)
    V_dot = 3 * V * s * c / (GAMMA * lam)
    b_dot = 4 * math.pi * G * GAMMA * (-(p_phi**2) / (2 * V**2) - 3 * s**2 / (8 * math.pi * G * GAMMA**2 * lam**2))
    phi_dot = p_phi / V
    return torch.stack([V_dot, b_dot, phi_dot], dim=-1)


def exact_volume(t: torch.Tensor, p_phi: torch.Tensor, lam: torch.Tensor) -> torch.Tensor:
    """V(t) = √(V_B² + 12πG p_φ² t²), t measured from the bounce."""
    V_B2 = p_phi**2 / (2 * critical_density(lam))
    return torch.sqrt(V_B2 + 12 * math.pi * G * p_phi**2 * t**2)


def time_to_bounce(y0: torch.Tensor, p_phi: torch.Tensor, lam: torch.Tensor) -> torch.Tensor:
    """Invert exact_volume: how long until V reaches V_B.   → (B,)"""
    V_B2 = p_phi**2 / (2 * critical_density(lam))
    return torch.sqrt(y0[:, 0] ** 2 - V_B2) / (math.sqrt(12 * math.pi * G) * p_phi)


def classical_volume(t: torch.Tensor, V0: torch.Tensor, p_phi: torch.Tensor) -> torch.Tensor:
    """GR: V̇ = −√(12πG) p_φ exactly, so V falls linearly and hits 0 (clamped there): the singularity."""
    return (V0 - math.sqrt(12 * math.pi * G) * p_phi * t).clamp(min=0)


def classical_singularity_time(V0, p_phi) -> float:
    """Time for classical contraction from V0 to V = 0 (ρ = ∞)."""
    return float(V0 / (math.sqrt(12 * math.pi * G) * p_phi))


def wdw_packet(x: torch.Tensor, phi: torch.Tensor, p_phi: float, x_b: float, spread: float, n_k: int = 401):
    r"""Ψ(φ, x) = Σ_k A(k) e^{ik(x − x_b + ln 2 + √(12πG) φ)}: a contracting Wheeler–DeWitt packet.   → (P, X) complex

    Positive frequency ω = √(12πG) k with k > 0 peaked at k₀ = p_φ/√(12πG) (ħ = 1), Gaussian of relative width
    spread. Each plane wave solves ∂²_φ Ψ = 12πG ∂²_x Ψ, so the packet moves rigidly along the classical
    trajectory x = x_b − ln 2 − √(12πG) φ, straight into x → −∞ (V → 0). The summed index is k.
    """
    k0 = p_phi / math.sqrt(12 * math.pi * G)
    k = torch.linspace(k0 * (1 - 6 * spread), k0 * (1 + 6 * spread), n_k, dtype=x.dtype, device=x.device)
    A = torch.exp(-((k - k0) ** 2) / (2 * (spread * k0) ** 2))  # (K,)
    arg = x[None, :, None] - x_b + math.log(2) + math.sqrt(12 * math.pi * G) * phi[:, None, None]  # (P, X, 1)
    waves = torch.polar(torch.ones_like(arg * k), arg * k)  # (P, X, K)
    return torch.einsum("k,pxk->px", A.to(waves.dtype), waves)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/cosmology/quantum_gravity_toy/tensors.py
"""Constants and initial states. Planck units: G = ħ = c = 1, so lengths in ℓ_P, times in t_P, densities in ρ_P.

The state of each universe in the batch is y_b = (V_b, b_b, φ_b): physical volume of a fixed comoving cell, the
connection variable b (conjugate to V, b ≈ γH classically) and the scalar field φ. Its momentum p_φ is conserved.
"""

import math

import torch

G = 1.0
GAMMA = 0.2375  # Barbero–Immirzi parameter (black-hole entropy counting)
LAMBDA = math.sqrt(4 * math.sqrt(3) * math.pi * GAMMA)  # λ² = Δ, the smallest non-zero area eigenvalue of LQG
RHO_C = 3 / (8 * math.pi * G * GAMMA**2 * LAMBDA**2)  # critical density ≈ 0.41 ρ_Planck


def initial_state(p_phi: torch.Tensor, lam: torch.Tensor, rho_start: float) -> torch.Tensor:
    """Contracting universes at density rho_start, on the constraint surface.   p_phi, lam: (B,) → y0 (B, 3)

    ρ = p_φ²/(2V²) fixes V; the constraint sin²(λb) = ρ/ρ_c fixes b, on the branch λb ∈ (π/2, π) where H < 0.
    """
    rho_c = 3 / (8 * math.pi * G * GAMMA**2 * lam**2)
    V0 = p_phi / math.sqrt(2 * rho_start)
    b0 = (math.pi - torch.asin(torch.sqrt(rho_start / rho_c))) / lam
    return torch.stack([V0, b0, torch.zeros_like(V0)], dim=-1)


def wdw_grid(x_min: float, x_max: float, phi_min: float, phi_max: float, nx: int, nphi: int, device: torch.device):
    """Grid of x = ln V and scalar-field clock φ for the Wheeler–DeWitt wave packet: (X,), (P,)."""
    x = torch.linspace(x_min, x_max, nx, dtype=torch.float64, device=device)
    phi = torch.linspace(phi_min, phi_max, nphi, dtype=torch.float64, device=device)
    return x, phi


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/cosmology/quantum_gravity_toy/solve.py
"""Integrate the effective LQC equations with RK4, a batch of universes in one tensor."""

import torch

from problems._common.integrators import integrate

from .equation import rhs


def evolve(y0: torch.Tensor, p_phi: torch.Tensor, lam: torch.Tensor, dt: float, t_span: float) -> dict:
    """y0 (B, 3) → {"t": (T,), "y": (T, B, 3)} over proper time [0, t_span]."""
    n = int(round(t_span / dt))
    y = integrate(lambda state, t: rhs(state, p_phi, lam), y0, dt, n)
    t = torch.arange(n + 1, dtype=y0.dtype, device=y0.device) * dt
    return {"t": t, "y": y}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/cosmology/quantum_gravity_toy/plot.py
"""Animate the LQC bounce next to classical collapse, the effective Friedmann curve and the Wheeler–DeWitt packet."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import G, classical_volume


def animate(sol, p, lam, scales, rho_c, H, rho, wdw, path: Path) -> Path:
    t = sol["t"].cpu().numpy()
    y = sol["y"].cpu().numpy()  # (T, B, 3)
    V, phi = y[..., 0], y[..., 2]
    Hn, rhon = H.cpu().numpy(), rho.cpu().numpy()
    rc = rho_c.cpu().numpy()
    V_cl = classical_volume(sol["t"], sol["y"][0, 0, 0], p[0]).cpu().numpy()
    with np.errstate(divide="ignore"):
        rho_cl = float(p[0]) ** 2 / (2 * V_cl**2)
    alive = V_cl > 0
    x, phi_grid, psi, x_b, phi_b = wdw
    prob = (psi.abs() ** 2).cpu().numpy()
    prob /= prob.max()
    labels = [f"LQC, λ × {s:.2g}" for s in scales.cpu().numpy()]
    shades = [ACCENT[0], "#60a5fa", "#93c5fd"]

    fig, ((ax_v, ax_r), (ax_h, ax_w)) = plt.subplots(2, 2, figsize=(8.6, 5.8))
    t_rel = t - t[np.argmin(V[:, 0])]

    for j in range(V.shape[1]):
        ax_v.plot(t_rel, V[:, j], color=shades[j], lw=1.6 if j == 0 else 1, label=labels[j])
        ax_r.plot(t_rel, rhon[:, j], color=shades[j], lw=1.6 if j == 0 else 1)
    ax_v.plot(t_rel[alive], V_cl[alive], "--", color=ACCENT[1], lw=1.3, label="classical GR")
    ax_r.plot(t_rel[alive], rho_cl[alive], "--", color=ACCENT[1], lw=1.3)
    ax_v.set_yscale("log")
    for a in (ax_v, ax_r):
        a.set_xscale("asinh", linear_width=0.2)  # linear near the bounce, logarithmic far away
        a.set_xticks([-3, -1, -0.3, 0, 0.3, 1, 3], ["−3", "−1", "−0.3", "0", "0.3", "1", "3"])
    ax_v.set_xlabel("proper time $t - t_B$ ($t_P$)")
    ax_v.set_ylabel("volume $V$ ($\\ell_P^3$)")
    ax_v.legend(fontsize=7, loc="upper center")
    ax_v.set_title("Big Bang → Big Bounce", fontsize=10)
    for j, r in enumerate(rc):
        ax_r.axhline(r, color=shades[j], ls=":", lw=1)
    ax_r.set_yscale("log")
    ax_r.set_ylim(rhon.min() * 0.5, rc.max() * 30)
    ax_r.set_xlabel("$t - t_B$ ($t_P$)")
    ax_r.set_ylabel("density $\\rho$ ($\\rho_P$)")
    ax_r.set_title("ρ stops at $\\rho_c$ (dotted); GR diverges", fontsize=10)
    dot_v = ax_v.plot([], [], "o", color="#111827", ms=5)[0]
    dot_r = ax_r.plot([], [], "o", color="#111827", ms=5)[0]

    # effective Friedmann: H² against ρ (classical straight line vs LQC parabolas)
    r_line = np.geomspace(rhon.min(), rc.max(), 300)
    ax_h.plot(r_line, 8 * math.pi * G / 3 * r_line, "--", color=ACCENT[1], lw=1.3, label="GR: $H^2=8\\pi G\\rho/3$")
    for j, r in enumerate(rc):
        rr = np.geomspace(rhon.min(), r, 400)
        ax_h.plot(rr, 8 * math.pi * G / 3 * rr * (1 - rr / r), color=shades[j], lw=1.6 if j == 0 else 1)
    ax_h.set_xscale("log")
    ax_h.set_xlim(rhon.min(), rc.max() * 1.2)
    ax_h.set_ylim(0, 8 * math.pi * G / 3 * rc[0] / 4 * 2.2)  # the full-gap parabola peaks at 2πGρ_c/3
    ax_h.set_xlabel("$\\rho$ ($\\rho_P$)")
    ax_h.set_ylabel("$H^2$ ($t_P^{-2}$)")
    ax_h.legend(fontsize=7, loc="upper left")
    ax_h.set_title("$H^2 = \\frac{8\\pi G}{3}\\rho(1-\\rho/\\rho_c)$ (simulated •)", fontsize=10)
    dot_h = ax_h.plot([], [], "o", color="#111827", ms=5)[0]

    # Wheeler–DeWitt |Ψ|² in the (φ, ln V) plane vs the effective LQC trajectory
    xs, ps = x.cpu().numpy(), phi_grid.cpu().numpy()
    ax_w.imshow(prob.T, origin="lower", aspect="auto", extent=(ps[0], ps[-1], xs[0], xs[-1]),
                cmap=plt.get_cmap("Greys", 6), vmin=0, vmax=1.2, interpolation="nearest")  # fmt: skip
    ax_w.plot(phi[:, 0], np.log(V[:, 0]), color=ACCENT[0], lw=1.6, label="LQC effective (bounces)")
    cl = x_b - math.log(2) - math.sqrt(12 * math.pi * G) * (ps - phi_b)
    ax_w.plot(ps, cl, "--", color=ACCENT[1], lw=1.2, label="GR contracting branch")
    ax_w.set_ylim(xs.min(), xs.max())
    ax_w.set_xlabel("scalar field clock $\\varphi$")
    ax_w.set_ylabel("$\\ln V$")
    ax_w.legend(fontsize=7, loc="upper center")
    ax_w.set_title("grey: Wheeler–DeWitt $|\\Psi(\\varphi, \\ln V)|^2$", fontsize=10)
    ax_w.grid(False)
    dot_w = ax_w.plot([], [], "o", color="#111827", ms=5)[0]
    label = fig.text(0.5, 0.005, "", ha="center", fontsize=9)

    def update(k: int):
        dot_v.set_data([t_rel[k]], [V[k, 0]])
        dot_r.set_data([t_rel[k]], [rhon[k, 0]])
        dot_h.set_data([rhon[k, 0]], [Hn[k, 0] ** 2])
        dot_w.set_data([phi[k, 0]], [np.log(V[k, 0])])
        phase = "contracting" if Hn[k, 0] < 0 else "expanding"
        label.set_text(
            f"t − t_B = {t_rel[k]:+.3f} t_P   ρ/ρ_c = {rhon[k, 0] / rc[0]:.4f}   H = {Hn[k, 0]:+.3f}  ({phase})"
        )

    fig.tight_layout(rect=(0, 0.03, 1, 1))
    return save_gif(fig, update, _frames(t_rel), path)


def _frames(t_rel: np.ndarray) -> list[int]:
    """Frames evenly spaced in asinh(t/t_B-width): dense near the bounce where everything happens."""
    u = np.arcsinh(t_rel / 0.2)
    targets = np.linspace(u[0], u[-1], len(frame_indices(len(u), 50)))
    return [int(i) for i in np.searchsorted(u, targets).clip(0, len(u) - 1)]


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'p_phi': 100.0,
    'gap_scale': 1.0,
    'log10_rho_start': -3.0,
    'dt': 0.002,
    'spread': 0.15
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.cosmology.quantum_gravity_toy")]:
    del sys.modules[name]
main = importlib.import_module("problems.cosmology.quantum_gravity_toy.main")
problem_dir = Path("problems/cosmology/quantum_gravity_toy")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
